# Lab 1: A Zero-Ops Vision + Tool-Calling Agent with Gemma 4 on Vertex AI OpenMaaS

**Open-Weight AI Models series, Part 2** · Kanchana Patlolla & Anusha Sabbineni · [openweights.cloudrace.info](https://openweights.cloudrace.info/)

> Unofficial practitioner guide, not official Google Cloud documentation.

This lab calls **Gemma 4 26B A4B IT** as a serverless managed API (`gemma-4-26b-a4b-it-maas`). There is no endpoint to deploy and no GPU to manage.

You will build a small "site scout" agent in three steps:

1. **See**: send an aerial or street photo and get labelled bounding boxes back, drawn on the image.
2. **Act**: let the model call two tools (place lookup and current weather) in a bounded agent loop.
3. **Show**: plot the tool results on an interactive map.

Per the [model page](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/maas/google/gemma-4-26b-a4b-it), this model accepts text and image input, supports function calling and structured output, is served from the `global` location, and is at the **Experimental** launch stage.

## Step 0: Setup

Before you run this:

1. Enable the Agent Platform API (`aiplatform.googleapis.com`) in your project.
2. Open the Gemma 4 model card in Model Garden and click **Enable**.
3. Set `PROJECT_ID` below.

Calls are billed pay-as-you-go to your project.

In [ ]:
%pip install --quiet openai google-auth pillow folium requests

import sys

PROJECT_ID = ""  # @param {type: "string"}
LOCATION = "global"
# OpenMaaS model names use the `publisher/model` form. If this name is rejected,
# copy the exact name from the model card's sample code.
MODEL = "google/gemma-4-26b-a4b-it-maas"  # @param {type: "string"}

assert PROJECT_ID, "Set PROJECT_ID first."

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)

import google.auth
import google.auth.transport.requests
import openai


def make_client() -> openai.OpenAI:
    """Returns an OpenAI client pointed at the OpenMaaS endpoint.

    Access tokens last about an hour; call this again if you get a 401.
    """
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(google.auth.transport.requests.Request())
    return openai.OpenAI(
        base_url=(
            f"https://aiplatform.googleapis.com/v1/projects/{PROJECT_ID}"
            f"/locations/{LOCATION}/endpoints/openapi"
        ),
        api_key=creds.token,
    )


client = make_client()
reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
    max_tokens=10,
    temperature=0,
)
print(reply.choices[0].message.content)

## Step 1: See — image in, bounding boxes out

Provide your own image: set `IMAGE_URL` to a public image, or leave it empty to upload a file in Colab. An aerial photo with clearly separated objects (buildings, vehicles, boats) works well on video.

**This cell is also the image-input test.** The image is sent as a base64 data URI in the OpenAI `image_url` format. If the endpoint rejects it, the error is printed and you can still run Step 2, which is text only.

Gemma 4 returns boxes as `box_2d: [y0, x0, y1, x1]` on a 0 to 1000 grid.

In [ ]:
import base64
import io
import json
import re

import requests
from PIL import Image, ImageDraw

IMAGE_URL = ""  # @param {type: "string"}
WHAT_TO_FIND = "buildings and vehicles"  # @param {type: "string"}

if IMAGE_URL:
    resp = requests.get(IMAGE_URL, timeout=30, headers={"User-Agent": "open-weight-models-lab"})
    resp.raise_for_status()
    image_bytes = resp.content
else:
    from google.colab import files
    uploaded = files.upload()
    image_bytes = next(iter(uploaded.values()))

image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
image.thumbnail((1024, 1024))
buf = io.BytesIO()
image.save(buf, format="JPEG", quality=90)
data_uri = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode("ascii")

detect_prompt = (
    f"Detect the {WHAT_TO_FIND} in this image. Return only a JSON list. Each item must have "
    '"box_2d": [y0, x0, y1, x1] normalized to 0-1000 and a short "label". At most 15 items.'
)


def parse_boxes(text: str) -> list[dict]:
    """Extracts the first JSON list from the model text and keeps well-formed boxes."""
    match = re.search(r"\[.*\]", text, flags=re.S)
    if not match:
        return []
    try:
        items = json.loads(match.group(0))
    except json.JSONDecodeError:
        return []
    boxes = []
    for item in items if isinstance(items, list) else []:
        box = item.get("box_2d") if isinstance(item, dict) else None
        if (
            isinstance(box, list)
            and len(box) == 4
            and all(isinstance(v, (int, float)) and 0 <= v <= 1000 for v in box)
            and box[0] < box[2]
            and box[1] < box[3]
        ):
            boxes.append({"box_2d": box, "label": str(item.get("label", ""))[:40]})
    return boxes


boxes = []
try:
    vision = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": data_uri}},
                {"type": "text", "text": detect_prompt},
            ],
        }],
        max_tokens=1500,
        temperature=0,
    )
    raw = vision.choices[0].message.content or ""
    print(raw)
    boxes = parse_boxes(raw)
except openai.APIError as e:
    print("Image request failed:", e)

annotated = image.copy()
draw = ImageDraw.Draw(annotated)
w, h = annotated.size
for b in boxes:
    y0, x0, y1, x1 = b["box_2d"]
    rect = [x0 / 1000 * w, y0 / 1000 * h, x1 / 1000 * w, y1 / 1000 * h]
    draw.rectangle(rect, outline="#ef4444", width=3)
    draw.text((rect[0] + 4, max(0, rect[1] - 14)), b["label"], fill="#ef4444")
print(f"{len(boxes)} boxes drawn")
annotated

## Step 2: Act — a bounded tool-calling agent

The model gets two tools and decides when to call them. Your code runs them; the model never executes anything itself.

- `find_place(name)`: looks up coordinates with the public Open-Meteo geocoding API.
- `get_current_weather(latitude, longitude)`: current temperature and wind from the public Open-Meteo forecast API.

Guardrails in the loop: tools are dispatched through an allowlist, arguments are validated, tool errors go back to the model as results, and the loop stops after `MAX_STEPS`.

In [ ]:
MAX_STEPS = 6
visited = []  # (name, lat, lon, weather) rows collected for the map


def find_place(name: str) -> dict:
    if not isinstance(name, str) or not 1 <= len(name) <= 100:
        return {"error": "name must be a string of 1-100 characters"}
    r = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": name, "count": 1, "language": "en", "format": "json"},
        timeout=20,
    )
    r.raise_for_status()
    results = r.json().get("results") or []
    if not results:
        return {"error": f"no match for {name!r}"}
    top = results[0]
    place = {
        "name": top.get("name"),
        "country": top.get("country"),
        "latitude": top.get("latitude"),
        "longitude": top.get("longitude"),
    }
    visited.append({**place, "weather": None})
    return place


def get_current_weather(latitude: float, longitude: float) -> dict:
    try:
        lat, lon = float(latitude), float(longitude)
    except (TypeError, ValueError):
        return {"error": "latitude and longitude must be numbers"}
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return {"error": "coordinates out of range"}
    r = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current": "temperature_2m,wind_speed_10m"},
        timeout=20,
    )
    r.raise_for_status()
    current = r.json().get("current") or {}
    weather = {
        "temperature_c": current.get("temperature_2m"),
        "wind_speed_kmh": current.get("wind_speed_10m"),
    }
    for row in visited:
        if abs(row["latitude"] - lat) < 0.01 and abs(row["longitude"] - lon) < 0.01:
            row["weather"] = weather
    return weather


TOOLS = {"find_place": find_place, "get_current_weather": get_current_weather}
ALLOWED_ARGS = {"find_place": {"name"}, "get_current_weather": {"latitude", "longitude"}}

TOOL_SPECS = [
    {
        "type": "function",
        "function": {
            "name": "find_place",
            "description": "Look up the latitude and longitude of a city or place by name.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string", "description": "Place name, e.g. Rotterdam"}},
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get current temperature (C) and wind speed (km/h) at coordinates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
            },
        },
    },
]


def run_tool(name: str, raw_args: str) -> dict:
    fn = TOOLS.get(name)
    if fn is None:
        return {"error": f"unknown tool {name}"}
    try:
        args = json.loads(raw_args or "{}")
    except json.JSONDecodeError:
        return {"error": "arguments were not valid JSON"}
    if not isinstance(args, dict) or set(args) - ALLOWED_ARGS[name]:
        return {"error": f"unexpected arguments for {name}"}
    try:
        return fn(**args)
    except Exception as e:  # report tool failures to the model instead of crashing
        return {"error": f"{type(e).__name__}: {e}"}


def run_agent(question: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a site scout. Use find_place to get coordinates, then "
                "get_current_weather for each place. Answer briefly using only tool results."
            ),
        },
        {"role": "user", "content": question},
    ]
    for step in range(MAX_STEPS):
        out = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SPECS, temperature=0, max_tokens=800
        )
        msg = out.choices[0].message
        if not msg.tool_calls:
            return msg.content or ""
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            result = run_tool(call.function.name, call.function.arguments)
            print(f"step {step + 1}: {call.function.name}({call.function.arguments}) -> {result}")
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)}
            )
    return "Stopped: reached MAX_STEPS without a final answer."


QUESTION = "Compare current conditions for a drone survey at the ports of Rotterdam, Singapore and Los Angeles. Which is calmest?"  # @param {type: "string"}
visited.clear()
print(run_agent(QUESTION))

## Step 3: Show — plot what the agent found

Each place the agent looked up becomes a marker, with the weather it fetched in the popup.

In [ ]:
import folium

if not visited:
    print("No places were looked up; run Step 2 first.")
else:
    center = [sum(r["latitude"] for r in visited) / len(visited),
              sum(r["longitude"] for r in visited) / len(visited)]
    fmap = folium.Map(location=center, zoom_start=2)
    for r in visited:
        wx = r["weather"] or {}
        popup = (
            f"{r['name']}, {r['country']}: {wx.get('temperature_c', '?')} C, "
            f"wind {wx.get('wind_speed_kmh', '?')} km/h"
        )
        folium.Marker([r["latitude"], r["longitude"]], popup=popup, tooltip=r["name"]).add_to(fmap)
    display(fmap)

## Next steps

- Swap `MODEL` for another OpenMaaS model and rerun Step 2 to compare tool-calling behaviour.
- Part 3 moves this loop into Google ADK with multi-model routing and verification.

Docs used: [Gemma 4 26B A4B IT model page](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/maas/google/gemma-4-26b-a4b-it) · [Call open model APIs](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/maas/call-open-model-apis)